# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a comprehensive guide for loading and exploring the FAIR² colorectal cancer dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. All dataset entities such as record sets, fields, and columns are referenced using their `@id` identifiers for clarity and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records from the Croissant schema using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Authors: {[a['@id'] for a in getattr(metadata, 'author', [])]}")

## 2. Data Overview
Let's review the available record sets and fields in this dataset schema. We access their `@id` for further extraction and analysis.

In [ ]:
# List all record sets referenced by @id in the dataset
from pprint import pprint

record_sets = []
if hasattr(metadata, 'recordSet'):
    if isinstance(metadata.recordSet, list):
        for r in metadata.recordSet:
            if isinstance(r, dict) and '@id' in r:
                record_sets.append(r['@id'])
            elif isinstance(r, str):
                record_sets.append(r)
    elif isinstance(metadata.recordSet, dict):
        record_sets.append(metadata.recordSet['@id'] if '@id' in metadata.recordSet else metadata.recordSet)

# If not present, attempt to discover record sets from dataset.records
if not record_sets:
    # mlcroissant provides dataset.record_sets
    try:
        record_sets = dataset.record_sets
    except Exception:
        pass

print("Available Record Sets by @id:")
for idx, rid in enumerate(record_sets):
    print(f"  {idx+1}. {rid}")
if record_sets:
    # Show fields for the first record set
    print(f"\nFields for record set '{record_sets[0]}':")
    fields = dataset.get_fields(record_sets[0])  # returns list of Field
    for f in fields:
        print(f"  - @id: {f['@id']}\tname: {f['name']}")
else:
    print("(No explicit record sets found in metadata. Attempting to list from .records())")

## 3. Data Extraction
We will extract tabular data from each available record set into a pandas DataFrame for exploration. All record set and field references are explicitly via their `@id`.

In [ ]:
# Extract data for each record set
# If record_sets is empty, try the default record set used in .records()
if not record_sets:
    # Fallback to discovering available record sets from dataset (mlcroissant >=0.4.2)
    try:
        # Show first found record set
        for rset in dataset.record_sets:
            record_sets.append(rset)
    except Exception:
        pass


dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for record set: {record_set_id}")
    records_iter = dataset.records(record_set=record_set_id)
    try:
        records = list(records_iter)
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")
        continue
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# Show column names for the first available DataFrame
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns in record set {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("No dataframes could be constructed. Please check dataset availability.")

## 4. Exploratory Data Analysis (EDA)

Here, we perform common data preparation and profiling. As an example, let's select an age-related or interval field (e.g. `age_at_second_crc_diagnosis`, if available), show some descriptive stats, filter the data, normalize values, and group by key clinicopathological variables like sex or anatomical location where possible.

In [ ]:
import numpy as np

# Use the main record set and guess numeric fields for demo
if dataframes:
    df_rs_id = list(dataframes.keys())[0]
    df = dataframes[df_rs_id].copy()
    possible_numeric_fields = [
        c for c in df.columns if any(
            kw.lower() in c.lower() for kw in ['age', 'interval', 'duration', 'years', 'months', 'count', 'number'])
           or np.issubdtype(df[c].dtype, np.number)
    ]
    if not possible_numeric_fields:
        possible_numeric_fields = df.select_dtypes('number').columns.tolist()
    
    # Choose a numeric field by preference or take the first
    if possible_numeric_fields:
        numeric_field = possible_numeric_fields[0]
    else:
        numeric_field = None
        print('No numeric fields found for EDA.')

    # Continue only if numeric_field is found
    if numeric_field:
        print(f"Analysis of numeric field (by @id): {numeric_field}\n")
        display(df[[numeric_field]].describe())
        threshold = df[numeric_field].mean() if not pd.isnull(df[numeric_field].mean()) else 0
        # Filter
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} rows")
        display(filtered_df[[numeric_field]].head(5))

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() else 1)
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Group by a likely categorical field (sex or location)
        possible_group_fields = [
            c for c in df.columns if any(kw in c.lower() for kw in ['sex', 'gender', 'location', 'group', 'site', 'anatomical'])
        ]
        if possible_group_fields:
            group_field = possible_group_fields[0]
            print(f"\nGrouping by categorical field (by @id): {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].agg(['mean', 'count'])
            display(grouped_df)
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No suitable numeric field found for EDA.")
else:
    print("No available dataframes for EDA.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field and its relationship to a key categorical group if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.histplot(df[numeric_field].dropna(), bins=15, ax=axes[0], kde=True)
    axes[0].set_title(f"Distribution of {numeric_field}")
    axes[0].set_xlabel(numeric_field)

    # Boxplot by group, if we have a grouping field
    if 'group_field' in locals() and group_field in df.columns:
        sns.boxplot(y=group_field, x=numeric_field, data=df, ax=axes[1])
        axes[1].set_title(f"{numeric_field} by {group_field}")
        axes[1].set_ylabel(group_field)
        axes[1].set_xlabel(numeric_field)
    else:
        axes[1].remove()
    plt.tight_layout()
    plt.show()
else:
    print("Visualization skipped - insufficient numeric data.")

## 6. Conclusion

In this notebook, we demonstrated how to use the `mlcroissant` library to explore a FAIR² clinical oncology dataset described by a Croissant schema. We loaded tabular data by record set `@id`, performed exploratory filtering and normalization based on available numeric fields, and visualized key data distributions. This approach ensures clarity and reproducibility in working with FAIR-compliant datasets. For further analysis, you may reference any entity using its `@id` from the metadata and dynamically extract additional record sets or fields as needed.